# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > 实践实验：用于基于内容过滤的深度学习

在本练习中，你将使用神经网络实现基于内容的过滤，来为电影构建一个推荐系统。


# 大纲
- [ 1 - 软件包 ](#1)
- [ 2 - 电影评分数据集 ](#2)
- [ 3 - 用神经网络进行基于内容的过滤](#3)
  - [ 3.1 训练数据](#3.1)
  - [ 3.2 准备训练数据](#3.2)
- [ 4 - 用于基于内容过滤的神经网络](#4)
  - [ 练习 1](#ex01)
- [ 5 - 预测](#5)
  - [ 5.1 - 为一个新用户做预测](#5.1)
  - [ 5.2 - 为一个已有用户做预测。](#5.2)
  - [ 5.3 - 找到相似的影片](#5.3)
    - [ 练习 2](#ex02)
- [ 6 - 恭喜完成！ ](#6)


_**注意：** 为防止自动评分器报错，你不得编辑或删除本实验中不参与评分的单元格。也请不要添加任何新单元格。
**一旦你通过了本次作业** 并想试验任何不参与评分的代码，你可以按照本笔记本底部的说明操作。_

<a name="1"></a>
## 1 - 软件包 <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
我们将使用熟悉的软件包 NumPy、TensorFlow，以及来自 [scikit-learn](https://scikit-learn.org/stable/) 的有用程序。我们还将使用 [tabulate](https://pypi.org/project/tabulate/) 来整齐地打印表格，使用 [Pandas](https://pandas.pydata.org/) 来组织表格数据。

In [1]:
import numpy as np
import numpy.ma as ma
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - 电影评分数据集 <img align="left" src="./images/film_rating.png" style=" width:40px;" >
该数据集源自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集。

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集大致有 600 位用户评分的 9000 部电影，评分以 0.5 到 5、步长 0.5 递增。该数据集在规模上已被缩减，以聚焦于 2000 年以来这些年份的电影和流行的类型。缩减后的数据集有 $n_u = 397$ 位用户，$n_m= 847$ 部电影和 25521 个评分。对每部电影，数据集提供一个电影标题、上映日期和一个或多个类型。例如"Toy Story 3"于 2010 年上映，有几个类型："Adventure|Animation|Children|Comedy|Fantasy"。这个数据集除了用户的评分之外，几乎不包含关于用户的其他信息。这个数据集用于为下面所述的神经网络创建训练向量。
让我们多了解一些这个数据集。下表显示了按评分数量排名的前 10 部电影。这些电影也恰好有较高的平均评分。这些电影中你看过多少部？

In [2]:
top10_df = pd.read_csv("./data/content_top10_df.csv")
bygenre_df = pd.read_csv("./data/content_bygenre_df.csv")
top10_df

,movie id,num ratings,ave rating,title,genres
0,4993,198,4.1,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
1,5952,188,4.0,"Lord of the Rings: The Two Towers, The",Adventure|Fantasy
2,7153,185,4.1,"Lord of the Rings: The Return of the King, The",Action|Adventure|Drama|Fantasy
3,4306,170,3.9,Shrek,Adventure|Animation|Children|Comedy|Fantasy|Ro...
4,58559,149,4.2,"Dark Knight, The",Action|Crime|Drama
5,6539,149,3.8,Pirates of the Caribbean: The Curse of the Bla...,Action|Adventure|Comedy|Fantasy
6,79132,143,4.1,Inception,Action|Crime|Drama|Mystery|Sci-Fi|Thriller
7,6377,141,4.0,Finding Nemo,Adventure|Animation|Children|Comedy
8,4886,132,3.9,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy
9,7361,131,4.2,Eternal Sunshine of the Spotless Mind,Drama|Romance|Sci-Fi


下一个表显示了按类型排序的信息。每个类型的评分数量差异很大。请注意，一部电影可能有多个类型，所以下面评分的总和大于原始评分的数量。

In [3]:
bygenre_df

,genre,num movies,ave rating/genre,ratings per genre
0,Action,321,3.4,10377
1,Adventure,234,3.4,8785
2,Animation,76,3.6,2588
3,Children,69,3.4,2472
4,Comedy,326,3.4,8911
5,Crime,139,3.5,4671
6,Documentary,13,3.8,280
7,Drama,342,3.6,10201
8,Fantasy,124,3.4,4468
9,Horror,56,3.2,1345


<a name="3"></a>
## 3 - 用神经网络进行基于内容的过滤

在协同过滤实验中，你生成了两个向量——一个用户向量和一个物品/电影向量，它们的点积会预测一个评分。这些向量完全从评分中推导出来。

基于内容的过滤也生成用户和电影特征向量，但它认识到可能还有关于用户和/或电影的其他可用信息，能够改进预测。这些额外信息被提供给一个神经网络，它随后生成用户向量和电影向量，如下所示。
<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>

<a name="3.1"></a>
### 3.1 训练数据
提供给网络的电影内容是原始数据与一些"工程化特征"（engineered features）的组合。回顾课程 1、第 2 周、实验 4 中的特征工程讨论和实验。原始特征是电影上映的年份和以 one-hot 向量呈现的电影类型。有 14 个类型。工程化特征是从用户评分推导出的平均评分。

用户内容由工程化特征构成。为每位用户计算每个类型的平均评分。此外，用户 id、评分计数和评分平均值也可用，但不包含在训练或预测内容中。它们随数据集一起携带，因为它们有助于解释数据。

训练集由数据集中用户所做的所有评分构成。有些评分被重复，以提升代表性不足类型的训练样本数量。训练集被拆分成两个具有相同条目数量的数组：一个用户数组和一个电影/物品数组。

在下面，让我们加载并显示一些数据。

In [4]:
# Load Data, set configuration variables
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
print(f"Number of training vectors: {len(item_train)}")

Number of training vectors: 50884


让我们看看用户训练数组中的前几个条目。

In [5]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9
2,22,4.0,4.0,4.2,0.0,0.0,4.0,4.1,4.0,4.0,0.0,3.0,4.0,0.0,3.9,3.9


一些用户和物品/电影特征在训练中没有被使用。在上面的表中，方括号 "[]" 中的特征，如 "user id"、"rating count" 和 "rating ave"，在模型被训练和使用时不包含在内。
在上面你可以看到用户 2 每个类型的评分平均值。为零的条目是该用户未曾评分的类型。用户向量对该用户评分的所有电影都是相同的。
让我们看看电影/物品数组的前几个条目。

In [6]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,1,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,1,0,1,0,0,0,0,0,1
46970,2006,3.2,1,0,0,0,1,0,0,0,0,0,0,0,0,0
48516,2006,4.3,0,0,0,0,0,1,0,1,0,0,0,0,0,1
58559,2008,4.2,1,0,0,0,0,1,0,1,0,0,0,0,0,0


在上面，电影数组包含影片上映的年份、平均评分，以及每个可能类型的一个指示符。对适用于该电影的每个类型，指示符为 1。电影 id 在训练中不使用，但在解释数据时有用。

In [7]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  3.5 4.  4.  4.5]


目标 y 是用户给出的电影评分。

在上面，我们可以看到电影 6874 是一部 2003 年上映的动作/犯罪/惊悚片。用户 2 给动作类电影的平均评分是 3.9。MovieLens 用户给这部电影的平均评分是 4。'y' 是 4，表明用户 2 也给电影 6874 评了 4 分。一个单独的训练样本由来自用户数组和物品数组各一行以及来自 y_train 的一个评分构成。

<a name="3.2"></a>
### 3.2 准备训练数据
回顾在课程 1、第 2 周中，你探索了特征缩放作为改进收敛的一种手段。我们将使用 [scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 来缩放输入特征。这在课程 1、第 2 周、实验 5 中用过。在下面，还展示了 inverse_transform 以产生原始输入。我们将使用一个 Min Max Scaler 来缩放目标评分，它把目标缩放到介于 -1 和 1 之间。[scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [8]:
# scale training data
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled    = y_train

scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

scalerTarget = MinMaxScaler((-1, 1))
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))
#ynorm_test = scalerTarget.transform(y_test.reshape(-1, 1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))

True
True


为了让我们能够评估结果，我们将像课程 2、第 3 周中讨论的那样，把数据拆分成训练集和测试集。这里我们将使用 [sklean train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来拆分并打乱数据。请注意，把初始随机状态设为相同的值，可确保 item、user 和 y 以相同方式被打乱。

In [9]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test data shape: {item_test.shape}")

movie/item training data shape: (40707, 17)
movie/item test data shape: (10177, 17)


缩放并打乱后的数据现在均值为零。

In [10]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
1,0,-1.0,-0.8,-0.7,0.1,-0.0,-1.2,-0.4,0.6,-0.5,-0.5,-0.1,-0.6,-0.6,-0.7,-0.7
0,1,-0.7,-0.5,-0.7,-0.1,-0.2,-0.6,-0.2,0.7,-0.5,-0.8,0.1,-0.0,-0.6,-0.5,-0.4
-1,-1,-0.2,0.3,-0.4,0.4,0.5,1.0,0.6,-1.2,-0.3,-0.6,-2.3,-0.1,0.0,0.4,-0.0
0,-1,0.6,0.5,0.5,0.2,0.6,-0.1,0.5,-1.2,0.9,1.2,-2.3,-0.1,0.0,0.2,0.3
-1,0,0.7,0.6,0.5,0.3,0.5,0.4,0.6,1.0,0.6,0.3,0.8,0.8,0.4,0.7,0.7


<a name="4"></a>
## 4 - 用于基于内容过滤的神经网络
现在，让我们构建一个如上图所述的神经网络。它将有两个网络，通过一个点积组合起来。你将构建这两个网络。在这个例子中，它们将是相同的。请注意，这些网络不需要相同。如果用户内容明显大于电影内容，你可能会选择增加用户网络相对于电影网络的复杂度。在这个例子中，内容是相似的，所以这些网络相同。

<a name="ex01"></a>
### 练习 1

- 使用一个 Keras 顺序（sequential）模型
    - 第一层是一个带 256 个单元和 relu 激活的 dense 层。
    - 第二层是一个带 128 个单元和 relu 激活的 dense 层。
    - 第三层是一个带 `num_outputs` 个单元和线性激活（或无激活）的 dense 层。

网络的其余部分将被提供。所提供的代码不使用 Keras 顺序模型，而是使用 Keras [函数式 api](https://keras.io/guides/functional_api/)。这种格式在组件如何互连方面允许更大的灵活性。


In [11]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(num_outputs, activation="linear")
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(num_outputs, activation="linear")  
  
    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features,))
vu = user_NN(input_user)
vu = keras.ops.normalize(vu, axis=1)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features,))
vm = item_NN(input_item)
vm = keras.ops.normalize(vm, axis=1)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32)        │     40,864 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32)        │     41,376 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize           │ (None, 32)        │          0 │ sequential[0][0]  │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_1         │ (None, 32)        │          0 │ sequential_1[0][… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ normalize[0][0],  │
│                     │                   │            │ normalize_1[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 82,240 (321.25 KB)

 Trainable params: 82,240 (321.25 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Public tests
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

All tests passed!
All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
  你可以像所示的那样创建一个带 relu 激活的 dense 层。
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看解答</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>

    

我们将使用均方误差损失和一个 Adam 优化器。

In [13]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [14]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], y_train, epochs=30)

Epoch 1/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 2s 909us/step - loss: 0.1233
Epoch 2/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 961us/step - loss: 0.1143
Epoch 3/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1089
Epoch 4/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 942us/step - loss: 0.1054
Epoch 5/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 965us/step - loss: 0.1024
Epoch 6/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 927us/step - loss: 0.0999
Epoch 7/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 932us/step - loss: 0.0979
Epoch 8/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 975us/step - loss: 0.0961
Epoch 9/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 906us/step - loss: 0.0942
Epoch 10/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 904us/step - loss: 0.0923
Epoch 11/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 885us/step - loss: 0.0907
Epoch 12/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 893us/step - loss: 0.0891
Epoch 13/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 974us/step - loss: 0.0877
Epoch 14/30
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 1s 937us/step - loss: 0.0864
Epo

评估模型以确定在测试数据上的损失。

In [15]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], y_test)

319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 643us/step - loss: 0.0847


0.08470921218395233

它与训练损失相当，表明模型没有明显对训练数据过拟合。

<a name="5"></a>
## 5 - 预测
在下面，你将在若干情形下使用你的模型来做预测。
<a name="5.1"></a>
### 5.1 - 为一个新用户做预测
首先，我们将创建一个新用户，并让模型为该用户建议电影。在你用示例用户内容试过之后，可以随意更改用户内容以匹配你自己的偏好，看看模型会建议什么。请注意评分介于 0.5 和 5.0 之间（含），以半步长递增。

In [16]:
new_user_id = 5000
new_rating_ave = 0.0
new_action = 0.0
new_adventure = 5.0
new_animation = 0.0
new_childrens = 0.0
new_comedy = 0.0
new_crime = 0.0
new_documentary = 0.0
new_drama = 0.0
new_fantasy = 5.0
new_horror = 0.0
new_mystery = 0.0
new_romance = 0.0
new_scifi = 0.0
new_thriller = 0.0
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

这个新用户喜欢冒险、奇幻类型的电影。让我们为这个新用户找到评分最高的电影。
在下面，我们将使用一组电影/物品向量 `item_vecs`，它为训练/测试集中的每部电影都有一个向量。这与上面的新用户向量相匹配，缩放后的向量被用来预测所有电影的评分。

In [17]:
# generate and replicate the user vector to match the number movies in the data set.
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# scale our user and item vectors
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

# make a prediction
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# unscale y prediction 
y_pu = scalerTarget.inverse_transform(y_p)

# sort the results, highest prediction first
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  #negate to get largest rating first
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  #using unscaled vectors for display

print_pred_movies(sorted_ypu, sorted_items, movie_dict, maxcount = 10)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


y_p,movie id,rating ave,title,genres
4.3,4896,3.8,Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001),Adventure|Children|Fantasy
4.3,137857,3.6,The Jungle Book (2016),Adventure|Drama|Fantasy
4.3,98809,3.8,"Hobbit: An Unexpected Journey, The (2012)",Adventure|Fantasy
4.3,81834,4,Harry Potter and the Deathly Hallows: Part 1 (2010),Action|Adventure|Fantasy
4.3,8368,3.9,Harry Potter and the Prisoner of Azkaban (2004),Adventure|Fantasy
4.2,40815,3.8,Harry Potter and the Goblet of Fire (2005),Adventure|Fantasy|Thriller
4.2,106489,3.6,"Hobbit: The Desolation of Smaug, The (2013)",Adventure|Fantasy
4.2,54001,3.9,Harry Potter and the Order of the Phoenix (2007),Adventure|Drama|Fantasy
4.2,59501,3.5,"Chronicles of Narnia: Prince Caspian, The (2008)",Adventure|Children|Fantasy
4.2,59387,4,"Fall, The (2006)",Adventure|Drama|Fantasy


<a name="5.2"></a>
### 5.2 - 为一个已有用户做预测。
让我们看看对"用户 2"（数据集中的用户之一）的预测。我们可以把预测评分与模型的评分作比较。

In [18]:
uid = 2 
# form a set of user vectors. This is the same vector, transformed and repeated.
user_vecs, y_vecs = get_user_vecs(uid, user_train_unscaled, item_vecs, user_to_genre)

# scale our user and item vectors
suser_vecs = scalerUser.transform(user_vecs)
sitem_vecs = scalerItem.transform(item_vecs)

# make a prediction
y_p = model.predict([suser_vecs[:, u_s:], sitem_vecs[:, i_s:]])

# unscale y prediction 
y_pu = scalerTarget.inverse_transform(y_p)

# sort the results, highest prediction first
sorted_index = np.argsort(-y_pu,axis=0).reshape(-1).tolist()  #negate to get largest rating first
sorted_ypu   = y_pu[sorted_index]
sorted_items = item_vecs[sorted_index]  #using unscaled vectors for display
sorted_user  = user_vecs[sorted_index]
sorted_y     = y_vecs[sorted_index]

#print sorted predictions for movies rated by the user
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, ivs, uvs, movie_dict, maxcount = 50)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


y_p,y,user,user genre ave,movie rating ave,movie id,title,genres
4.6,5.0,2,[4.0],4.3,80906,Inside Job (2010),Documentary
4.3,4.0,2,"[4.0,4.1,4.0,4.0,3.9,3.9]",4.1,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller
4.3,3.5,2,"[4.0,4.1,4.0,3.9]",3.8,8798,Collateral (2004),Action|Crime|Drama|Thriller
4.2,3.5,2,"[4.0,4.0]",3.9,99114,Django Unchained (2012),Action|Drama
4.2,4.0,2,"[4.0,4.1,3.9]",4.0,6874,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
4.2,4.5,2,"[4.0,4.0]",4.1,68157,Inglourious Basterds (2009),Action|Drama
4.1,5.0,2,"[4.0,4.1,4.0]",3.9,106782,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
4.1,4.5,2,"[4.0,4.1,4.0]",4.2,58559,"Dark Knight, The (2008)",Action|Crime|Drama
4.0,4.0,2,"[4.1,4.0,3.9]",4.3,48516,"Departed, The (2006)",Crime|Drama|Thriller
3.9,3.5,2,"[4.0,3.9,3.9]",3.9,115713,Ex Machina (2015),Drama|Sci-Fi|Thriller


模型预测通常在实际评分的 1 分以内，尽管它对用户如何给特定电影评分并不是一个非常准确的预测器。如果用户评分与该用户的类型平均值差异显著，尤其如此。你可以在上面更改用户 id 来尝试不同的用户。并非所有用户 id 都被用在了训练集中。

<a name="5.3"></a>
### 5.3 - 找到相似的影片
上面的神经网络产生两个特征向量：一个用户特征向量 $v_u$，和一个电影特征向量 $v_m$。这些是 32 项的向量，其值难以解释。然而，相似的影片会有相似的向量。这个信息可以用来做推荐。例如，如果一个用户给"Toy Story 3"评了高分，就可以通过选择具有相似电影特征向量的电影来推荐相似的电影。

一个相似度度量是两个向量 $\mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ 之间的平方距离：
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex02"></a>
### 练习 2

编写一个函数来计算平方距离。

In [19]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    ### START CODE HERE ###     
    d = np.sum(np.square(a-b))
    ### END CODE HERE ###     
    return d

In [20]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1):0.3f}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2):0.3f}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3):0.3f}")

squared distance between a1 and b1: 0.000
squared distance between a2 and b2: 0.030
squared distance between a3 and b3: 2.000


**预期输出**：

squared distance between a1 and b1: 0.000    
squared distance between a2 and b2: 0.030   
squared distance between a3 and b3: 2.000

In [21]:
# Public tests
test_sq_dist(sq_dist)

All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
  虽然求和往往意味着应使用一个 for 循环，但这里的减法可以在一条语句中逐元素进行。此外，你可以利用 np.square 来对减法的结果逐元素求平方。np.sum 可用于对这些平方后的元素求和。
    
</details>

    

一旦模型训练好，电影之间的距离矩阵可以计算一次，然后在无需重新训练的情况下被重复用于新的推荐。模型训练好后的第一步，是为每部电影获得电影特征向量 $v_m$。为此，我们将使用训练好的 `item_NN`，并构建一个小模型，让我们能够把电影向量传入其中以生成 $v_m$。

In [22]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features,))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
vm_m = keras.ops.normalize(vm_m, axis=1)                        # incorporate normalization as was done in the original model
model_m = tf.keras.Model(input_item_m, vm_m)                                
model_m.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 32)             │        41,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalize_2 (Normalize)         │ (None, 32)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,376 (161.62 KB)

 Trainable params: 41,376 (161.62 KB)

 Non-trainable params: 0 (0.00 B)

一旦你有了一个电影模型，你就可以通过使用一组物品/电影向量作为输入、用该模型进行预测，来创建一组电影特征向量。`item_vecs` 是所有电影向量的一组。它必须经过缩放才能与训练好的模型一起使用。预测的结果是每部电影一个 32 项的特征向量。

In [23]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
size of all predicted movie feature vectors: (847, 32)


现在让我们计算每个电影特征向量与所有其他电影特征向量之间平方距离的一个矩阵：
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>

然后我们可以通过沿每一行找到最小值来找到最接近的电影。我们将利用 [numpy 掩码数组](https://numpy.org/doc/1.21/user/tutorial-ma.html) 来避免选中同一部电影。对角线上被掩码的值不会被包含在计算中。

In [24]:
count = 50  # number of movies to display
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    disp.append( [movie_dict[movie1_id]['title'], movie_dict[movie1_id]['genres'],
                  movie_dict[movie2_id]['title'], movie_dict[movie1_id]['genres']]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow")
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama|Romance,Mona Lisa Smile (2003),Drama|Romance
"Wedding Planner, The (2001)",Comedy|Romance,Mr. Deeds (2002),Comedy|Romance
Hannibal (2001),Horror|Thriller,Final Destination 2 (2003),Horror|Thriller
Saving Silverman (Evil Woman) (2001),Comedy|Romance,"Sweetest Thing, The (2002)",Comedy|Romance
Down to Earth (2001),Comedy|Fantasy|Romance,Bewitched (2005),Comedy|Fantasy|Romance
"Mexican, The (2001)",Action|Comedy,Rush Hour 2 (2001),Action|Comedy
15 Minutes (2001),Thriller,Panic Room (2002),Thriller
Enemy at the Gates (2001),Drama,Seabiscuit (2003),Drama
Heartbreakers (2001),Comedy|Crime|Romance,Fun with Dick and Jane (2005),Comedy|Crime|Romance
Spy Kids (2001),Action|Adventure|Children|Comedy,Scooby-Doo (2002),Action|Adventure|Children|Comedy


结果表明，模型通常会建议一部具有相似类型的电影。

<a name="6"></a>
## 6 - 恭喜完成！ <img align="left" src="./images/film_award.png" style=" width:40px;">
你已经完成了一个基于内容的推荐系统。

这种结构是许多商业推荐系统的基础。如果有可用信息，用户内容可以大大扩展，以纳入关于用户的更多信息。物品不限于电影。这可以用来推荐任何物品——书籍、汽车，或与你"购物车"中某件物品相似的物品。

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想试验任何不参与评分的代码，请点击这里。</b></font></summary>
    <p><i><b>重要提示：请仅在你已经通过本次作业之后再这样做，以避免自动评分器出现问题。</b></i>
    <ol>
        <li> 在笔记本的菜单上，点击 "View" > "Cell Toolbar" > "Edit Metadata"</li>
        <li> 点击你想锁定/解锁的代码单元格旁边的 "Edit Metadata" 按钮</li>
        <li> 把 "editable" 的属性值设为：
            <ul>
                <li> 如果你想解锁它，设为 "true" </li>
                <li> 如果你想锁定它，设为 "false" </li>
            </ul>
        </li>
        <li> 在笔记本的菜单上，点击 "View" > "Cell Toolbar" > "None" </li>
    </ol>
    <p> 这里有一个简短的演示，展示如何执行上述步骤：
        <br>
        <img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>